In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from tqdm import tqdm

## Парсинг новостей

In [3]:
def parse_news_for_ticker(ticker):
    """
    Парсит все новости для заданного тикера с сайта Smart-Lab.
    Возвращает список словарей с полями: raw_date, headline, news_url, ticker.
    """
    base_url = f"https://smart-lab.ru/forum/news/{ticker}/page"
    news_data = []
    page = 1  # Начиная с первой страницы

    while True:
        url = f"{base_url}{page}/"
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Ошибка при запросе страницы {url}: {response.status_code}")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        news_block = soup.find('div', class_='temp_block')
        if not news_block:
            # Если блок новостей отсутствует, прекращаем парсинг
            break

        news_list = news_block.find('ul', class_='temp_headers temp_headers--have-numbers')
        if not news_list:
            break

        news_items = news_list.find_all('li')
        if not news_items:
            break

        for item in news_items:
            # Извлечение даты
            date_section = item.find('b')
            if not date_section:
                continue

            # Дата между `</b>` и `<a>`
            raw_date = date_section.next_sibling.strip()

            # Извлечение заголовка и ссылки
            headline_tag = item.find('a')
            if not headline_tag:
                continue

            headline = headline_tag.get_text(strip=True)
            news_url = headline_tag.get('href', '').strip()

            # Сохраняем данные
            news_data.append({
                "raw_date": raw_date,
                "headline": headline,
                "news_url": news_url,
                "ticker": ticker
            })

        page += 1  # Переходим на следующую страницу

    return news_data

In [22]:
tickers = ['CBOM', 'ALRS', 'VTBR', 'MDMG', 'GEMC', 'VKCO', 'LENT', 'RUAL', 'T', 'HEAD', 'ENPG', 'YDEX', 'BSPB', 'AQUA',
           'AFKS', 'AFLT', 'VSEH', 'GAZP', 'GMKN', 'LSRG', 'POSI', 'RENI', 'EUTR', 'IRAO', 'X5', 'LEAS', 'MVID',
           'MBNK', 'MAGN', 'MTLR', 'MTLRP', 'MTSS', 'MOEX', 'LKOH', 'BELU', 'NLMK', 'OZPH', 'PIKK', 'PLZL', 'VSMO',
           'RTKM', 'RTKMP', 'SBER', 'SBERP', 'CHMF', 'SELG', 'SVCB', 'FLOT', 'TGKA', 'TRNFP', 'HYDR', 'FEES', 'PHOR',
           'ELFV', 'SFIN', 'UPRO', 'ASTR', 'SGZH', 'RNFT', 'MSNG', 'SMLT', 'NVTK', 'ROSN', 'TATN', 'TATNP', 'PRMD',
           'HNFG', 'AKRN', 'APTK', 'ABIO', 'WUSH', 'OGKB', 'DATA', 'FESH', 'DIAS', 'IVAT', 'KMAZ', 'DELI', 'RASP', 'MRKV',
           'MSRS', 'MRKZ', 'MRKS', 'MRKU', 'MRKP', 'MRKC', 'SVAV', 'SOFL', 'SNGS', 'SNGSP', 'TGKN', 'TRMK', 'UGLD', 'VSMO',
           'PRMD', 'HNFG', 'AKRN', 'APTK', 'ABIO', 'WUSH', 'OGKB', 'DATA', 'FESH', 'DIAS', 'IVAT', 'KMAZ', 'DELI', 'RASP',
           'MRKV', 'MSRS', 'MRKZ', 'MRKS', 'MRKU', 'MRKP', 'MRKC', 'SVAV', 'SOFL', 'SNGS', 'SNGSP', 'TGKN', 'TRMK', 'UGLD']

tickers = list(set(tickers))

print(tickers)

['ROSN', 'TGKA', 'VSMO', 'RUAL', 'EUTR', 'RENI', 'LEAS', 'PLZL', 'PRMD', 'KMAZ', 'TGKN', 'RASP', 'MRKU', 'SFIN', 'WUSH', 'DELI', 'SNGSP', 'HEAD', 'T', 'AQUA', 'CHMF', 'SGZH', 'RTKMP', 'IRAO', 'DATA', 'NLMK', 'MRKC', 'SVAV', 'MVID', 'MSRS', 'SOFL', 'YDEX', 'AFLT', 'MBNK', 'SVCB', 'FLOT', 'LSRG', 'SNGS', 'OZPH', 'HNFG', 'LENT', 'ELFV', 'TRMK', 'POSI', 'MDMG', 'PHOR', 'FEES', 'AKRN', 'RNFT', 'ABIO', 'OGKB', 'GMKN', 'RTKM', 'SMLT', 'GEMC', 'X5', 'ALRS', 'TATNP', 'DIAS', 'MAGN', 'HYDR', 'TRNFP', 'MOEX', 'CBOM', 'UPRO', 'PIKK', 'SBERP', 'VKCO', 'MRKS', 'NVTK', 'MRKZ', 'VSEH', 'GAZP', 'SBER', 'APTK', 'MSNG', 'IVAT', 'MRKP', 'MTLR', 'MTLRP', 'VTBR', 'MRKV', 'FESH', 'UGLD', 'MTSS', 'TATN', 'AFKS', 'LKOH', 'SELG', 'ENPG', 'BSPB', 'BELU', 'ASTR']


In [15]:
# Список для тикеров, по которым нет новостей
no_news_tickers = []

# Итоговый DataFrame для всех тикеров
all_news = []

# Прогресс-бар для тикеров
with tqdm(total=len(tickers), desc="Обработка тикеров") as pbar:
    for ticker in tickers:
        news = parse_news_for_ticker(ticker)

        if news:
            all_news.extend(news)
        else:
            no_news_tickers.append(ticker)

        pbar.update(1)  # Обновляем прогресс-бар

Обработка тикеров: 100%|█████████████████████████████████████████████████████████████████| 1/1 [00:16<00:00, 16.31s/it]


In [16]:
# Сохранение данных в CSV
if all_news:
    news_df = pd.DataFrame(all_news)
    news_df.to_csv("news2.csv", index=False)
    print("Все новости сохранены в 'news.csv'")
else:
    print("Новостей не найдено.")

# Вывод тикеров без новостей
if no_news_tickers:
    print("Тикеры, для которых не найдены новости:")
    print(", ".join(no_news_tickers))
else:
    print("Новости найдены для всех тикеров.")


Все новости сохранены в 'news.csv'
Новости найдены для всех тикеров.


In [19]:
news = pd.read_csv("news.csv")
news.head()

,raw_date,headline,news_url,ticker
0,06/01,Администрация Байдена планирует усилить санкци...,/blog/1101455.php,ROSN
1,28/12,Меняем рекомендацию по бумагам Роснефти c поку...,/blog/1099995.php,ROSN
2,27/12,СД Роснефти решил продлить полномочия Сечина н...,/blog/1099673.php,ROSN
3,27/12,"📈Котировки Роснефти и Русснефти +3% и 3,4% соо...",/blog/1099565.php,ROSN
4,26/12,Акционеры Роснефти одобрили дивиденды за 9м 20...,/blog/1099257.php,ROSN


In [23]:
tickers_news = (list(news['ticker'].unique()))

In [24]:
for ticker in tickers:
    if ticker not in tickers_news:
        print(ticker)

SNGSP
RTKMP
X5
TATNP
SBERP
MTLRP


In [25]:
# Загрузка данных
file_path = "news.csv"  # Путь к файлу
news_data = pd.read_csv(file_path)

In [26]:
# Тикеры для дублирования и их новые значения
original_tickers = ["SNGS", "RTKM", "TATN", "SBER", "MTLR"]
new_tickers = [ticker + "P" for ticker in original_tickers]

In [27]:
# Фильтрация данных по тикерам
filtered_data = news_data[news_data["ticker"].isin(original_tickers)].copy()

In [28]:
# Дублирование записей с новыми тикерами
for original, new in zip(original_tickers, new_tickers):
    # Создаем копию записей и заменяем тикер
    duplicate_data = filtered_data[filtered_data["ticker"] == original].copy()
    duplicate_data["ticker"] = new
    # Добавляем в основной DataFrame
    news_data = pd.concat([news_data, duplicate_data], ignore_index=True)

In [29]:
# Сохранение обновленного файла
news_data.to_csv(file_path, index=False)
print(f"Обновленный файл сохранен: {file_path}")

Обновленный файл сохранен: news.csv


In [30]:
# Установка текущей даты
current_date = datetime(2025, 1, 8)

In [31]:
# Функция для преобразования raw_date
def process_dates(group):
    year = current_date.year  # Начальный год
    previous_month = None

    def convert_date(raw_date):
        nonlocal year, previous_month

        if ":" in raw_date:  # Формат hh:mm
            return current_date.strftime("%Y-%m-%d")

        if "/" in raw_date:  # Формат DD/MM
            day, month = map(int, raw_date.split("/"))
            # Устанавливаем начальный год на основе первой записи
            if previous_month is None:
                if month in [11, 12]:
                    year -= 1
            elif month > previous_month:  # Месяц больше предыдущего, уменьшаем год
                year -= 1

            previous_month = month
            return f"{year}-{month:02d}-{day:02d}"

        # Если формат не соответствует, возвращаем raw_date как есть (на случай ошибок)
        return raw_date

    # Преобразуем даты в группе
    group["date"] = group["raw_date"].apply(convert_date)
    return group

In [32]:
# Преобразуем даты для каждой группы тикеров
news_data = news_data.groupby("ticker", group_keys=False).apply(process_dates)

C:\Users\andre\AppData\Local\Temp\ipykernel_16076\1733918323.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  news_data = news_data.groupby("ticker", group_keys=False).apply(process_dates)


In [33]:
# Сохраняем обновленный файл
news_data.to_csv(file_path, index=False)
print(f"Файл успешно обновлен и сохранен: {file_path}")

Файл успешно обновлен и сохранен: news.csv


## Фильтрация новостей

In [2]:
from datetime import timedelta

In [2]:
# Параметры
file_path = "news.csv"  # Исходный файл
filtered_file_path = "news_filtered.csv"  # Результирующий файл
days_gap = 1  # Разрыв в днях
strategy = 2  # 1 - удаление цепочки, 2 - удаление только текущей новости

In [4]:
# Загрузка данных
news_data = pd.read_csv(file_path)
news_data["date"] = pd.to_datetime(news_data["date"])  # Преобразуем даты в формат datetime

In [5]:
# Добавляем индекс для сохранения исходного порядка
news_data = news_data.reset_index()

In [6]:
# Функция для фильтрации новостей по тикерам
def filter_news_by_ticker(group):
    # Сортируем новости только по дате для проверки условий
    sorted_group = group.sort_values(by="date").reset_index(drop=True)

    filtered_indices = []  # Индексы для "свободных" новостей
    last_removed_date = None  # Последняя убранная дата (для strategy = 1)

    for i, row in sorted_group.iterrows():
        current_date = row["date"]

        # Находим ближайшие даты до и после текущей новости
        prev_date = sorted_group.loc[i - 1, "date"] if i > 0 else None
        next_date = sorted_group.loc[i + 1, "date"] if i < len(sorted_group) - 1 else None

        # Проверяем условия для стратегии
        is_free = True

        if prev_date is not None:
            is_free &= current_date >= prev_date + timedelta(days=days_gap)

        if next_date is not None:
            is_free &= current_date <= next_date - timedelta(days=days_gap)

        if not is_free:
            if strategy == 1:
                last_removed_date = current_date
            continue  # Пропускаем текущую новость (удаляем)

        # Strategy 1: Удаляем цепочку новостей, связанных с убранной
        if strategy == 1 and last_removed_date is not None and current_date <= last_removed_date:
            continue  # Пропускаем текущую новость

        # Добавляем индекс новости, если она "свободна"
        filtered_indices.append(row["index"])

    return filtered_indices

In [7]:
# Применяем фильтрацию по каждому тикеру
filtered_indices = []
for ticker, group in news_data.groupby("ticker"):
    filtered_indices.extend(filter_news_by_ticker(group))

In [8]:
# Сохраняем только "свободные" новости, сохраняя исходный порядок
filtered_data = news_data[news_data["index"].isin(filtered_indices)].sort_values(by="index")

In [9]:
# Убираем технический индекс
filtered_data.drop(columns=["index"], inplace=True)

In [10]:
# Сохраняем результат
filtered_data.to_csv(filtered_file_path, index=False)
print(f"Фильтрация завершена. Результат сохранен в '{filtered_file_path}'.")

Фильтрация завершена. Результат сохранен в 'news_filtered.csv'.


## Обработка полных новостей

In [5]:
df = pd.read_csv('news.csv')
df.head()

,raw_date,headline,news_url,ticker,date
0,06/01,Администрация Байдена планирует усилить санкци...,/blog/1101455.php,ROSN,2025-01-06
1,28/12,Меняем рекомендацию по бумагам Роснефти c поку...,/blog/1099995.php,ROSN,2024-12-28
2,27/12,СД Роснефти решил продлить полномочия Сечина н...,/blog/1099673.php,ROSN,2024-12-27
3,27/12,"📈Котировки Роснефти и Русснефти +3% и 3,4% соо...",/blog/1099565.php,ROSN,2024-12-27
4,26/12,Акционеры Роснефти одобрили дивиденды за 9м 20...,/blog/1099257.php,ROSN,2024-12-26


In [10]:
df.columns

Index(['raw_date', 'headline', 'news_url', 'ticker', 'date'], dtype='object')

In [11]:
grouped = df.groupby(['ticker', 'date', 'raw_date'], as_index=False).agg({
    'headline': ' '.join,
    'news_url': ' '.join
})
grouped.to_csv('news_grouped.csv', index=False)

## Добавляем цены на тикер и индекс

In [12]:
import pandas as pd
from datetime import timedelta

In [13]:
# Параметры
news_file = "news_grouped.csv"
ticker_data_dir = "data_ticker/"
index_file = "data_index/IMOEX_data.csv"

In [14]:
# Загрузка данных
news_data = pd.read_csv(news_file)
imoex_data = pd.read_csv(index_file, parse_dates=["TRADEDATE"])
imoex_data = imoex_data.sort_values(by="TRADEDATE")

In [15]:
# Кэш для данных тикеров
ticker_data_cache = {}

In [16]:
def load_ticker_data(ticker):
    """Загружает данные для указанного тикера и кэширует их."""
    if ticker not in ticker_data_cache:
        ticker_file = f"{ticker_data_dir}{ticker}_data.csv"
        ticker_data_cache[ticker] = pd.read_csv(ticker_file, parse_dates=["TRADEDATE"]).sort_values(by="TRADEDATE")
    return ticker_data_cache[ticker]

In [17]:
def get_nearest_price(data, target_date, direction="current"):
    """
    Получает ближайшую цену для указанной даты.
    direction: "current" (до или на дату), "before", "after".
    """
    if direction == "current":
        nearest_data = data[data["TRADEDATE"] <= target_date]
    elif direction == "before":
        nearest_data = data[data["TRADEDATE"] < target_date]
    elif direction == "after":
        nearest_data = data[data["TRADEDATE"] > target_date]
    else:
        raise ValueError("Invalid direction specified.")
    
    if not nearest_data.empty:
        if direction == "after":
            return nearest_data.iloc[0]["CLOSE"]
        else:
            return nearest_data.iloc[-1]["CLOSE"]
    return None

In [18]:
# Создаем новые столбцы и заполняем их
news_data["Price_DG_before"] = None
news_data["Price_current"] = None
news_data["Price_DG_after"] = None
news_data["IMOEX_DG_before"] = None
news_data["IMOEX_current"] = None
news_data["IMOEX_DG_after"] = None

In [19]:
# Обработка строк с прогресс-баром
with tqdm(total=len(news_data), desc="Вычисление цен") as pbar:
    for index, row in news_data.iterrows():
        ticker = row["ticker"]
        news_date = pd.to_datetime(row["date"])

        # Загрузка данных тикера
        ticker_data = load_ticker_data(ticker)

        # Вычисляем цены для актива
        news_data.at[index, "Price_current"] = get_nearest_price(ticker_data, news_date, "current")
        news_data.at[index, "Price_DG_before"] = get_nearest_price(ticker_data, news_date, "before")
        news_data.at[index, "Price_DG_after"] = get_nearest_price(ticker_data, news_date, "after")

        # Вычисляем цены для IMOEX
        news_data.at[index, "IMOEX_current"] = get_nearest_price(imoex_data, news_date, "current")
        news_data.at[index, "IMOEX_DG_before"] = get_nearest_price(imoex_data, news_date, "before")
        news_data.at[index, "IMOEX_DG_after"] = get_nearest_price(imoex_data, news_date, "after")

        pbar.update(1)  # Обновляем прогресс-бар

# Сохраняем результат
news_data.to_csv("news_grouped_with_prices.csv", index=False)
print("Обновленный файл сохранен как 'news_with_prices.csv'.")

Вычисление цен:  29%|█████████████████▍                                         | 14639/49655 [00:30<01:11, 487.02it/s]


KeyboardInterrupt: 